# Kaggle house prices - attempt

the ames housing dataset. target: low rmse on the public leaderboard.

plan: lots of feature engineering on a small dataset, stacked models.


In [ ]:
import pandas as pd
import numpy as np

tr = pd.read_csv('data/house-prices/train.csv')
te = pd.read_csv('data/house-prices/test.csv')
print(tr.shape, te.shape)
print(tr['SalePrice'].describe())


In [ ]:
# log target since prices are skewed
tr['logSalePrice'] = np.log1p(tr['SalePrice'])
tr['logSalePrice'].hist(bins=50)


In [ ]:
# missing value heatmap
import seaborn as sns
missing = tr.isnull().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:
# fill numeric with median, categorical with 'None'
num = tr.select_dtypes(include=[np.number]).columns
cat = tr.select_dtypes(include=['object']).columns
for c in num:
    tr[c] = tr[c].fillna(tr[c].median())
    te[c] = te[c].fillna(tr[c].median()) if c in te else None
for c in cat:
    tr[c] = tr[c].fillna('None')
    te[c] = te[c].fillna('None')


In [ ]:
# ordinal encoding for ordered cats
quality_map = {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0}
for c in ['ExterQual','ExterCond','BsmtQual','HeatingQC','KitchenQual']:
    tr[c] = tr[c].map(quality_map).fillna(0)
    te[c] = te[c].map(quality_map).fillna(0)


In [ ]:
# new features
for df in [tr, te]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)


In [ ]:
# dummies for the rest
combined = pd.concat([tr.drop(columns='SalePrice'), te])
combined = pd.get_dummies(combined)
tr_X = combined.iloc[:len(tr)]
te_X = combined.iloc[len(tr):]
print(tr_X.shape, te_X.shape)
